# Классическое ML (sklearn)

Линейные модели, KNN, SVM, деревья, Random Forest, кластеризация, PCA, Pipeline + GridSearch.

## Импорты

In [1]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, mean_squared_error, r2_score
from sklearn.datasets import load_iris

## Подготовка данных

In [ ]:
# X, y -- уже готовые numpy/pandas (после feature engineering из tabular.ipynb)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y  # stratify убрать для регрессии
)

## Линейные модели

In [ ]:
from sklearn.linear_model import LinearRegression, Ridge, Lasso, LogisticRegression

# --- Регрессия ---
pipe_ridge = Pipeline([
    ('scaler', StandardScaler()),
    ('model', Ridge(alpha=1.0)),
])
pipe_ridge.fit(X_train, y_train)
preds = pipe_ridge.predict(X_test)
print(f'Ridge RMSE: {mean_squared_error(y_test, preds, squared=False):.4f}')

# Lasso (L1, разреживает признаки)
# pipe_lasso = Pipeline([('scaler', StandardScaler()), ('model', Lasso(alpha=0.1))])

# --- Классификация ---
pipe_logreg = Pipeline([
    ('scaler', StandardScaler()),
    ('model', LogisticRegression(C=1.0, max_iter=1000)),
])
pipe_logreg.fit(X_train, y_train)
preds = pipe_logreg.predict(X_test)
print(classification_report(y_test, preds))

## KNN

In [ ]:
from sklearn.neighbors import KNeighborsClassifier, KNeighborsRegressor

pipe_knn = Pipeline([
    ('scaler', StandardScaler()),  # KNN чувствителен к масштабу!
    ('model', KNeighborsClassifier(n_neighbors=5, metric='euclidean')),
])
pipe_knn.fit(X_train, y_train)
preds = pipe_knn.predict(X_test)
print(classification_report(y_test, preds))

# Подбор K
# param_grid = {'model__n_neighbors': [3, 5, 7, 11, 15, 21]}
# gs = GridSearchCV(pipe_knn, param_grid, cv=5, scoring='f1_macro')
# gs.fit(X_train, y_train)
# print(f'Best K: {gs.best_params_}, F1: {gs.best_score_:.4f}')

## SVM

In [ ]:
from sklearn.svm import SVC, SVR

pipe_svm = Pipeline([
    ('scaler', StandardScaler()),  # SVM тоже чувствителен к масштабу
    ('model', SVC(kernel='rbf', C=1.0, gamma='scale', probability=True)),
])
pipe_svm.fit(X_train, y_train)
preds = pipe_svm.predict(X_test)
print(classification_report(y_test, preds))

# probability=True нужен для predict_proba (например для AUC-ROC)
# proba = pipe_svm.predict_proba(X_test)

## Деревья и Random Forest

In [ ]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier

# Decision Tree
dt = DecisionTreeClassifier(max_depth=10, random_state=42)
dt.fit(X_train, y_train)
print('Decision Tree:')
print(classification_report(y_test, dt.predict(X_test)))

# Random Forest
rf = RandomForestClassifier(n_estimators=300, max_depth=15, random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)
print('Random Forest:')
print(classification_report(y_test, rf.predict(X_test)))

# sklearn Gradient Boosting (медленнее CatBoost/XGBoost, но иногда полезен)
# gb = GradientBoostingClassifier(n_estimators=300, learning_rate=0.1, max_depth=5)
# gb.fit(X_train, y_train)

## Кластеризация

In [ ]:
from sklearn.cluster import KMeans, DBSCAN
from sklearn.metrics import silhouette_score

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# --- K-Means ---
kmeans = KMeans(n_clusters=3, random_state=42, n_init=10)
labels_km = kmeans.fit_predict(X_scaled)
print(f'KMeans silhouette: {silhouette_score(X_scaled, labels_km):.4f}')

# Подбор числа кластеров по inertia (elbow method)
# inertias = []
# for k in range(2, 15):
#     km = KMeans(n_clusters=k, random_state=42, n_init=10)
#     km.fit(X_scaled)
#     inertias.append(km.inertia_)
# plt.plot(range(2, 15), inertias, marker='o')
# plt.xlabel('K'); plt.ylabel('Inertia'); plt.show()

# --- DBSCAN ---
dbscan = DBSCAN(eps=0.5, min_samples=5)
labels_db = dbscan.fit_predict(X_scaled)
n_clusters = len(set(labels_db)) - (1 if -1 in labels_db else 0)
print(f'DBSCAN clusters: {n_clusters}, noise points: {(labels_db == -1).sum()}')

## PCA

In [ ]:
from sklearn.decomposition import PCA
import matplotlib.pyplot as plt

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Снижение размерности с сохранением 95% дисперсии
pca = PCA(n_components=0.95)
X_pca = pca.fit_transform(X_scaled)
print(f'Было {X.shape[1]} признаков -> стало {X_pca.shape[1]}')
print(f'Объяснённая дисперсия: {pca.explained_variance_ratio_.sum():.4f}')

# Визуализация в 2D
pca2d = PCA(n_components=2)
X_2d = pca2d.fit_transform(X_scaled)
plt.figure(figsize=(8, 6))
plt.scatter(X_2d[:, 0], X_2d[:, 1], c=y, cmap='viridis', alpha=0.5, s=10)
plt.xlabel('PC1'); plt.ylabel('PC2')
plt.colorbar(label='target')
plt.title('PCA 2D projection')
plt.show()

## Pipeline + GridSearchCV

In [ ]:
pipe = Pipeline([
    ('scaler', StandardScaler()),
    ('model', RandomForestClassifier(random_state=42)),
])

param_grid = {
    'model__n_estimators': [100, 300, 500],
    'model__max_depth': [5, 10, 15, None],
}

gs = GridSearchCV(
    pipe,
    param_grid,
    cv=5,
    scoring='f1_macro',  # f1_macro / accuracy / neg_mean_squared_error
    n_jobs=-1,
    verbose=1,
)
gs.fit(X_train, y_train)

print(f'Best params: {gs.best_params_}')
print(f'Best CV score: {gs.best_score_:.4f}')
print(classification_report(y_test, gs.predict(X_test)))